# Combinatorial Optimization with QAOA

The Quantum Approximate Optimization Algorithm (QAOA) is a variational
algorithm for combinatorial optimization problems. It alternates layers of:

- **Cost unitary** exp(-i * gamma * C) encoding the objective function
- **Mixer unitary** exp(-i * beta * B) exploring the solution space

This notebook shows how to define optimization problems, build QAOA circuits,
and analyze how circuit resources scale with problem size and circuit depth.

In [ ]:
from qdk_pythonic.domains.optimization import MaxCut, QUBO, QAOA

## Max-Cut Problem

Given a graph, Max-Cut finds a partition of the nodes into two sets that
maximizes the number of edges crossing the partition. The cost Hamiltonian
encodes each edge as a ZZ interaction:

$$C = \sum_{(i,j) \in E} \frac{1 - Z_i Z_j}{2}$$

In [ ]:
# Triangle graph: 3 nodes, 3 edges
triangle = MaxCut(edges=[(0, 1), (1, 2), (2, 0)], n_nodes=3)
ham = triangle.to_hamiltonian()

print(f"Nodes: {triangle.n_nodes}")
print(f"Edges: {len(triangle.edges)}")
print(f"Hamiltonian terms: {len(ham)}")
print(f"Qubits: {ham.qubit_count()}")

## QAOA Circuit

The QAOA builder takes a cost Hamiltonian and a depth parameter `p`.
It constructs a circuit with `p` alternating cost/mixer layers, starting
from a uniform superposition.

In [ ]:
qaoa = QAOA(cost_hamiltonian=ham, p=1)
circuit = qaoa.to_circuit(gamma=[0.5], beta=[0.7])

print(f"Parameters: {qaoa.num_parameters} (2 * p = 2 * {qaoa.p})")
print(f"Total gates: {circuit.total_gate_count()}")
print(f"Depth: {circuit.depth()}")
print()
print(circuit.draw())

## QAOA Depth Scaling

Increasing `p` improves the approximation ratio but adds more gates.
Each layer contributes one cost unitary and one mixer unitary.

In [ ]:
print(f"{'p':>4}  {'Params':>8}  {'Gates':>8}  {'Depth':>8}")
print("-" * 34)

for p in [1, 2, 3, 4]:
    q = QAOA(cost_hamiltonian=ham, p=p)
    gamma = [0.5] * p
    beta = [0.7] * p
    c = q.to_circuit(gamma=gamma, beta=beta)
    print(f"{p:>4}  {q.num_parameters:>8}  {c.total_gate_count():>8}  {c.depth():>8}")

## Larger Problem: Petersen Graph

The Petersen graph has 10 nodes and 15 edges, making it a good benchmark
for a mid-size Max-Cut instance.

In [ ]:
# Petersen graph edges
petersen_edges = [
    (0, 1), (1, 2), (2, 3), (3, 4), (4, 0),  # outer cycle
    (5, 7), (7, 9), (9, 6), (6, 8), (8, 5),  # inner star
    (0, 5), (1, 6), (2, 7), (3, 8), (4, 9),  # spokes
]

petersen = MaxCut(edges=petersen_edges, n_nodes=10)
ham_p = petersen.to_hamiltonian()

qaoa_p = QAOA(cost_hamiltonian=ham_p, p=2)
circ_p = qaoa_p.to_circuit(gamma=[0.5, 0.3], beta=[0.7, 0.2])

print(f"Nodes: {petersen.n_nodes}, Edges: {len(petersen.edges)}")
print(f"Hamiltonian terms: {len(ham_p)}")
print(f"QAOA p=2: {circ_p.total_gate_count()} gates, depth {circ_p.depth()}")

## QUBO Formulation

Many real-world optimization problems are naturally expressed as QUBO
(Quadratic Unconstrained Binary Optimization): minimize x^T Q x.
The QUBO class converts this to an Ising Hamiltonian via x_i = (1 - Z_i) / 2.

In [ ]:
# A 4-variable QUBO with diagonal and off-diagonal terms
Q = {
    (0, 0): -1.0,  # linear term on x_0
    (1, 1): -2.0,  # linear term on x_1
    (2, 2): -1.0,
    (3, 3): -1.5,
    (0, 1): 1.0,   # quadratic coupling x_0 * x_1
    (1, 2): 0.5,
    (2, 3): 1.5,
}

qubo = QUBO(Q=Q, n_vars=4)
ham_q = qubo.to_hamiltonian()

print(f"QUBO variables: {qubo.n_vars}")
print(f"QUBO entries: {len(qubo.Q)}")
print(f"Ising Hamiltonian terms: {len(ham_q)}")

# Build a QAOA circuit from the QUBO
qaoa_q = QAOA(cost_hamiltonian=ham_q, p=2)
circ_q = qaoa_q.to_circuit(gamma=[0.4, 0.6], beta=[0.3, 0.5])
print(f"QAOA p=2: {circ_q.total_gate_count()} gates, depth {circ_q.depth()}")

## Resource Estimation

> **Note:** The cell below requires `qsharp>=1.25` (`pip install qsharp>=1.25`).

Estimate the physical resources for running QAOA on the Petersen graph.

In [ ]:
# Requires: pip install qsharp>=1.25
# result = qaoa_p.estimate_resources(
#     gamma=[0.5, 0.3], beta=[0.7, 0.2],
# )
# print(result)